# Stage 4 — Domain-Adapt Fine-Tune: Multi-Task (Qwen3-VL)

Self-contained. This is the **deferred task-neutral domain-adaptation pass** (separate
checkpoint from the combined detection fine-tune) — branches fresh from the original
Qwen3-VL base, per the 2026-07-10 resequencing decision.

**Three mixed task types**, so the model isn't overfit to one question framing:
1. **OCR/tag-reading** (Gupta) — real Tesseract OCR output, used as machine-generated
   pseudo-labels (Gupta has no real tag-transcription ground truth — checked the full
   archive directly, no CSV/JSON text labels anywhere).
2. **Symbol counting** (Gupta) — real ground-truth box counts, no OCR involved.
3. **Typed symbol summary** (Kaggle) — real 32-class detection labels via `classes.json`,
   broadens domain exposure beyond Gupta-only and beyond text-reading.

A 4th task (connectivity/relations from PID2Graph) is deferred — that download isn't
complete yet; revisit once this run is done and benchmarked.

**Built to run unattended overnight:** checkpoints to Drive every epoch, resumes from
the latest checkpoint automatically if the runtime disconnects and you reconnect and
re-run this notebook.

## 1. Mount Drive (needed for overnight checkpoint persistence)

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

MessageError: User cancelled dfs_ephemeral authorization

## 2. Install dependencies (transformers stack + Tesseract OCR)

In [ ]:
!apt-get -qq install -y tesseract-ocr
!pip install -q transformers accelerate peft pytesseract qwen-vl-utils

## 3. Config + imports

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pid_project/data"
GUPTA_DIR = "gupta_pid"
KAGGLE_DIR = "kaggle_pid_symbols"
CHECKPOINT_DIR = "/content/drive/MyDrive/pid_project/checkpoints/qwen3vl_domain_adapt"

import json, time, random
from pathlib import Path
import torch
import pytesseract
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

ROOT = Path(DRIVE_ROOT)
gupta_p = ROOT / GUPTA_DIR
kaggle_p = ROOT / KAGGLE_DIR
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

classes_path = ROOT / "classes.json"
class_names = {}
if classes_path.exists():
    _cdata = json.loads(classes_path.read_text())
    class_names = {cid: entry["kaggle_name"] for cid, entry in _cdata["classes"].items()}

print("Gupta:", gupta_p.exists(), "| Kaggle:", kaggle_p.exists(), "| classes.json:", classes_path.exists())
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — check Runtime type before continuing"

## 4. Build the mixed training pool: OCR tags + symbol counts (Gupta) + typed summary (Kaggle)

In [ ]:
TILE_SIZE, OVERLAP = 1024, 205

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            tiles.append((x0, y0, x1, y1))
            x0 += stride
        y0 += stride
    return tiles

def yolo_line_to_xyxy(line, img_w, img_h):
    parts = line.split()
    cls_id = parts[0]
    cx, cy, w, h = (float(v) for v in parts[1:5])
    cx, cy, w, h = cx * img_w, cy * img_h, w * img_w, h * img_h
    return cls_id, [cx - w/2, cy - h/2, cx + w/2, cy + h/2]

def boxes_intersect(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    return ax0 < bx1 and ax1 > bx0 and ay0 < by1 and ay1 > by0

def ocr_tags(tile_img, min_len=2):
    """Real Tesseract OCR — output is a pseudo-label, not verified ground truth."""
    raw_text = pytesseract.image_to_string(tile_img)
    tokens = [t.strip() for t in raw_text.split() if len(t.strip()) >= min_len]
    tags = [t for t in tokens if any(c.isdigit() for c in t) or "-" in t]
    return sorted(set(tags))


# --- Task 1 + 2: Gupta OCR tags + symbol counts, same tiling pass ---
OCR_PROMPT = "List every text tag or label you can read in this P&ID tile."
COUNT_PROMPT = "How many distinct symbols (valves, instruments, fittings, etc.) are visible in this P&ID tile?"

def build_gupta_examples(max_sheets=72):
    raw = gupta_p / "PID_Dataset" / "0__raw_data"
    sheet_paths = sorted((raw / "sheets" / "train").glob("*.*"))[:max_sheets]
    examples = []
    for sheet_path in sheet_paths:
        label_path = raw / "labels" / "train" / f"{sheet_path.stem}.txt"
        img = Image.open(sheet_path).convert("RGB")
        W, H = img.size
        orig_boxes = []
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                if line.strip():
                    orig_boxes.append(yolo_line_to_xyxy(line, W, H)[1])
        for (x0, y0, x1, y1) in compute_tile_grid(W, H):
            tbox = (x0, y0, x1, y1)
            n_boxes = sum(1 for b in orig_boxes if boxes_intersect(b, tbox))
            crop = img.crop(tbox)

            tags = ocr_tags(crop)
            if tags:
                target = "Tags visible in this tile: " + ", ".join(tags)
                examples.append({"task": "ocr", "image_path": str(sheet_path), "crop": [x0, y0, x1, y1],
                                  "prompt": OCR_PROMPT, "target_text": target})

            count_target = (
                f"There are {n_boxes} symbols visible in this tile." if n_boxes != 1
                else "There is 1 symbol visible in this tile."
            )
            examples.append({"task": "count", "image_path": str(sheet_path), "crop": [x0, y0, x1, y1],
                              "prompt": COUNT_PROMPT, "target_text": count_target})
    return examples


# --- Task 3: Kaggle typed symbol summary ---
TYPED_SUMMARY_PROMPT = "What types of symbols are visible in this P&ID tile, and how many of each?"

def build_kaggle_examples(max_images=6591):
    from collections import Counter
    examples = []
    label_files = sorted((kaggle_p / "labels").glob("*.txt"))[:max_images]
    for lbl in label_files:
        lines = [l for l in lbl.read_text().splitlines() if l.strip()]
        if not lines:
            continue
        counts = Counter(l.split()[0] for l in lines)
        parts = []
        for cls_id, n in counts.most_common():
            name = class_names.get(cls_id, f"class_{cls_id}")
            parts.append(f"{name} x{n}")
        target = "This tile contains: " + ", ".join(parts) + "."
        img_path = kaggle_p / "images" / f"{lbl.stem}.jpg"
        examples.append({"task": "typed_summary", "image_path": str(img_path), "crop": None,
                          "prompt": TYPED_SUMMARY_PROMPT, "target_text": target})
    return examples


print("Building Gupta OCR + counting examples (scans all 72 train sheets — may take a few minutes)...")
t0 = time.time()
gupta_examples = build_gupta_examples()
print(f"Built {len(gupta_examples)} Gupta examples in {time.time()-t0:.1f}s "
      f"({sum(1 for e in gupta_examples if e['task']=='ocr')} ocr, "
      f"{sum(1 for e in gupta_examples if e['task']=='count')} count)")

print("\nBuilding Kaggle typed-summary examples (using a sample, not all 30k — keeps the pool balanced "
      "across task types rather than dominated by Kaggle volume)...")
kaggle_examples = build_kaggle_examples(max_images=1000)
print(f"Built {len(kaggle_examples)} Kaggle typed-summary examples")

all_examples = gupta_examples + kaggle_examples
random.seed(0)
random.shuffle(all_examples)
print(f"\nTotal mixed training pool: {len(all_examples)} examples")
print("Sample:", all_examples[0])

## 5. Load Qwen3-VL, apply LoRA

In [ ]:
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
OCR_PROMPT = "List every text tag or label you can read in this P&ID tile."

processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda")
print("Base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Resume from latest checkpoint, if any

Checkpoints save every `CHECKPOINT_EVERY_N_STEPS` steps (not just per epoch) — resumes
from the exact step within an epoch, not just the last completed epoch, so an overnight
disconnect loses at most a few hundred steps, not a whole epoch.

In [ ]:
CHECKPOINT_EVERY_N_STEPS = 200

ckpt_path = Path(CHECKPOINT_DIR)
latest_ckpt_dir = ckpt_path / "latest"
progress_file = ckpt_path / "progress.json"

start_epoch, start_step = 0, 0
if progress_file.exists() and latest_ckpt_dir.exists():
    progress = json.loads(progress_file.read_text())
    start_epoch = progress["epoch"]
    start_step = progress["step"] + 1
    model.load_adapter(str(latest_ckpt_dir), adapter_name="default", is_trainable=True)
    print(f"Resumed from {latest_ckpt_dir}, continuing at epoch {start_epoch} step {start_step}")
else:
    print("No existing checkpoint — starting from epoch 0 step 0")

## 7. Training loop — checkpoints every 200 steps, safe to leave running overnight

Wraps each step in try/except so one bad example (e.g. a corrupt OCR result) can't kill
an unattended multi-hour run. Saves the rolling "latest" checkpoint + progress.json every
CHECKPOINT_EVERY_N_STEPS steps, so a disconnect loses at most that many steps, not a
whole epoch. Named epoch_N snapshots are also kept at each epoch boundary for history.

In [ ]:
N_EPOCHS = 3
LR = 1e-4

def build_labeled_inputs(image, prompt, target_text, image_crop=None):
    if image_crop is not None:
        image = image.crop(image_crop)
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    prompt_inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    )
    prompt_len = prompt_inputs["input_ids"].shape[1]
    full_messages = messages + [{"role": "assistant", "content": [{"type": "text", "text": target_text}]}]
    full_inputs = processor.apply_chat_template(
        full_messages, tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt",
    )
    labels = full_inputs["input_ids"].clone()
    labels[:, :prompt_len] = -100
    full_inputs["labels"] = labels
    return {k: v.to(model.device) for k, v in full_inputs.items()}

def save_checkpoint(epoch, step):
    model.save_pretrained(str(latest_ckpt_dir))
    progress_file.write_text(json.dumps({"epoch": epoch, "step": step}))

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)

for epoch in range(start_epoch, N_EPOCHS):
    random.seed(epoch)
    order = list(range(len(all_examples)))
    random.shuffle(order)
    skip_to = start_step if epoch == start_epoch else 0

    epoch_losses = {"ocr": [], "count": [], "typed_summary": []}
    t0 = time.time()
    for step, idx in enumerate(order):
        if step < skip_to:
            continue
        ex = all_examples[idx]
        try:
            img = Image.open(ex["image_path"]).convert("RGB")
            inputs = build_labeled_inputs(img, ex["prompt"], ex["target_text"], image_crop=ex.get("crop"))
            out = model(**inputs)
            loss = out.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            epoch_losses[ex["task"]].append(loss.item())
        except Exception as e:
            task_name = ex["task"]
            print(f"  [skip] step {step} ({task_name}) failed: {e}")
            continue

        if step % 50 == 0:
            all_losses = [l for losses in epoch_losses.values() for l in losses]
            avg = sum(all_losses[-50:]) / len(all_losses[-50:]) if all_losses else float("nan")
            print(f"epoch {epoch} step {step}/{len(order)} avg_loss={avg:.4f} elapsed={time.time()-t0:.0f}s")

        if step % CHECKPOINT_EVERY_N_STEPS == 0 and step > 0:
            save_checkpoint(epoch, step)
            print(f"  [checkpoint] saved at epoch {epoch} step {step}")

    # end-of-epoch: save both the rolling "latest" (for resume) and a named snapshot (history)
    save_checkpoint(epoch, len(order) - 1)
    epoch_ckpt = ckpt_path / f"epoch_{epoch}"
    model.save_pretrained(str(epoch_ckpt))
    print(f"=== epoch {epoch} done, saved to {epoch_ckpt} (and latest/) ===")
    for task, losses in epoch_losses.items():
        if losses:
            print(f"  {task:15s} n={len(losses):5d} mean_loss={sum(losses)/len(losses):.4f}")
    print()

print("Training complete.")